# Physics 152 - Final Project: Higgs Mass Reconstruction
## UCSC - Winter 2024
## Juan Robles
## Physics 152

## Introduction

Applications of Machine Learning (ML) onto other scientific fields has been prevalent over the past decades and has made its way onto fields such as     particle physics. Within this field ML has been used as a means of identifying particles that are massive, implying a short life span [[1]](https://hdsr.mitpress.mit.edu/pub/xqle7lat/release5#:~:text=Machine%20learning%20(ML%20has%20played,with%20a%20primordial%20bottom%20quark.), while also being used for event identification and mass reconstruction [[2]](https://iopscience.iop.org/article/10.1088/1742-6596/1085/2/022008/pdf). The focus of this project was to construct a Machine Learning Algorithm (MLA) that allowed the machine to learn a non-linear problem, that is, it learned how to calculate the invariant mass of a particle from input data. Since this is a regression problem, the MLA was written with the goal of solving this sort     of problem. In order to understand what exactly the machine is doing we must understand relativistic kinematics and collider coordinates.

### Particle Physics Primer

#### Relativistic Kinematics

Due to the particle's high speed we must resort to using relativity in order to accurately describe their motion through space. This implies we must work with four-dimensional vectors, in this case, momentum-four-vectors [[3]](https://mikefragugliacom.files.wordpress.com/2016/12/introduction-to-elementary-particles-gnv64.pdf). Consider the following,

\begin{equation}
\textbf{p}^{\mu} = \left(\frac{E}{c}, \textbf{p} \right),
\tag{1}
\end{equation}
where $\textbf{p}^{\mu}$ denotes the four-momentum vector, E is the energy of the particle in question, and $\textbf{p}$ is the three dimensional momentum vector of the particle [[3]](https://mikefragugliacom.files.wordpress.com/2016/12/introduction-to-elementary-particles-gnv64.pdf). Since momentum must be invariant its scalar product yields [[3]](https://mikefragugliacom.files.wordpress.com/2016/12/introduction-to-elementary-particles-gnv64.pdf),

\begin{equation}
\textbf{p}_{\mu}\textbf{p}^{\mu} = \left(\frac{E}{c}\right)^{2} - p^{2} = \left(mc\right)^2
\tag{2}
\end{equation}
where $ \textbf{p}_{\mu}\textbf{p}^{\mu}$ is simply the scalar product of a four dimensional vector.

To motivate the intuition, consider a particle with four-momentum $\textbf{p}_1$, that decays into two different particles with their respective momenta; $\textbf{p}_2$, and $\textbf{p}_3$. Lets consider the case where the daughter particles shoot out in opposite directions, such that, conservation of momentum gives: $\textbf{p}_1 = \textbf{p}_2 - \textbf{p}_3$.

Taking the scalar product of this vector:
\begin{align}
p_1^2 &= p_2^2 + p_3^2 - 2\textbf{p}_2\cdot\textbf{p}_3\\
\left(m_1c\right)^2 &= \left(m_2c\right)^2 + \left(m_3c\right)^2 -2\left( \frac{E_2E_3}{c^2}\right)\\
m_1^2 &= m_2^2 + m_3^2 -\frac{2E_2E_3}{c^4}\\
m_1 &= \sqrt{m_2^2 + m_3^2 -\frac{2E_2E_3}{c^4}}\\
\end{align}

Thus showing that we can obtain the mass of the parent particle from the four-momentum vector of the daughter particles. I could also leave the final solution in terms of momentum and energy,

\begin{equation}
m_1 = \frac{1}{c}\sqrt{p_2^2 + p_3^2 -\frac{2E_2E_3}{c^2}}.
\tag{3}
\end{equation}
Equation (3) is much more useful when your measurements involve momentum and energy *not* mass. However, the need for a coordinate transformation may aid in the calculations of invariant masses, as is the case in particle colliders.

#### Collider Coordinates


Consider a beam that is going along the z-axis such that the xy-plane is transverse to the beam axis; in such a configuration we can then define the transverse momentum as [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1),
\begin{equation}
\textbf{p}_{T} = \left(\textbf{p}_x, \textbf{p}_y\right)
\end{equation}
However, we generally call this the transverse momentum [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1),
\begin{equation}
p_T = \sqrt{p_x^2+p_y^2}
\tag{4}
\end{equation}

Since the collider will be boosting the particles that compose the beams, we want to choose coordinates that are invariant under the Lorentz boost [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1). However, the z-momentum and the Energy "mix" under the Lorentz boost, making it necessary to define rapidity, a quantity that is invariant under the Lorentz boost,
\begin{equation}
y = \frac{1}{2}\ln\left(\frac{E+p_z}{E-p_z}\right).
\end{equation}
Within colliders, it so happens that the energy is much greater than the mass of the particles, so we can approximate the z-momentum as, $p_z\approx E\cos\left(\theta\right)$, where the angle $\theta$ is measured relative to the beam axis [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1). Therefore, we can define pseudo-rapidity as,
\begin{equation}
\eta = -\ln\left(\tan\left(\frac{\theta}{2}\right)\right).
\tag{5}
\end{equation}
This new quantity helps see that if the angle theta gets bigger, the particles are less likely to be produces in the initial collision [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1).



Finally the angle $\phi$ is in the transverse plane and thus it will be invariant under the Lorentz boost [[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1). The invariant mass of the particle in question can be obtained, using these collider coordinates, with the following expression [[5]](https://en.wikipedia.org/wiki/Invariant_mass#Collider_experiments),

\begin{equation}
m = \sqrt{2p_{T1}p_{T2}\left[\cosh\left(\eta_{1}-\eta_{2}\right)-\cos\left(\phi_{1}-\phi_{2} \right) \right]}
\tag{6}
\end{equation}

With this new set of coordinates and an expression that can be used to solve for the invariant mass we can now read and filter the data.

## Data

### Data Filtering

The data set was provided by Professor Hance, and it holds events in which there was a Higgs Boson decaying to two muons. In order to read the data file, the *h5py* package was important and was used to read data files with an h5py extension. In order to faciliate this process, I imported *pandas* and I created a class that focuses on reading the data and returns the target data as *m* and the data file with the same size as the data set *m*.

In [ ]:
import h5py
import pandas as pd


class ReadsData:
    """
    Takes in the following inputs:
        + Data file.
        + The branches that I am interested in
        + The target data file.
    Returns:
        + Data set that is the same size as the target data set.
        + Target data set; true mass.
    """

    def __init__(self, dataFile, branches, targetFile):
        self.data = dataFile
        self.branches = branches
        self.target = targetFile

    # Splits the data file into the respective variables
    def DataRead(self):
        # Makes it easy to read the branches and the target files
        branches = self.branches
        target = self.target
        with h5py.File(self.data, "r") as hdf5file:
            data = hdf5file[list(hdf5file.keys())[0]]["lowleveltree"]
            alldata = data[branches]
        # Data is now the extracted data
        # m is the target data without nan-values
        data = pd.DataFrame(alldata)
        m = pd.read_csv(target)
        m = m.dropna(axis=0)
        # Returns the target data and the feed through data with same size.
        return data[0 : len(m)], m

The class above will return the target data set, and the input data set with both having the same length.

In [ ]:
# Name of the input data file:
dataFile = "lowlevelAna_Hmumu.hf5"
# Branches that are of interest to me:
branches = (
    "numjet",
    "numlepton",
    "numbtagjet",
    "met",
    "metphi",
    "lepton1pT",
    "lepton1eta",
    "lepton1phi",
    "lepton2pT",
    "lepton2eta",
    "lepton2phi",
    "jet1pT",
    "jet1eta",
    "jet1phi",
    "jet1b",
    "jet2pT",
    "jet2eta",
    "jet2phi",
    "jet2b",
    "jet3pT",
    "jet3eta",
    "jet3phi",
    "jet3b",
    "jet4pT",
    "jet4eta",
    "jet4phi",
    "jet4b",
    "jet5pT",
    "jet5eta",
    "jet5phi",
    "jet5b",
    "jet6pT",
    "jet6eta",
    "jet6phi",
    "jet6b",
)
# Target data file; true masses.
targetFile = "targets_Hmumu.txt"
# Initializing my ReadsData object:
trialOne = ReadsData(dataFile, branches, targetFile)
# Defining the variables that will hold the output of DataRead()
data, m = trialOne.DataRead()

Lets take a look at how the input data set is formated:

In [ ]:
data

The shape of my data lets me know that it has 35 *features*, and 49,999 occurances for each feature. Notice that these features are defined in terms of the collider coordinates we derived in the previous section. These features let us know that the machine will be fed information that is collider coordinates, and it will have the job of figuring out the relationship between these features by considering the target data.

Based on the shape of my data, I decided to define the following parameters:
+ test size = 10,000
+ batch size = 49,999
+ input number = 35

With these parameters in mind I decided to write a class called *LearningHiggs*, in which I defined a function that would sort my data into training and test batches.

In [ ]:
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import torch.nn
import torch.nn.functional
import torch.optim
import torch.utils.data


class LearningHiggs:
    def __init__(self, X, Y, test_size, batch_size, input_n):
        self.X = X
        self.Y = Y
        self.test_size = test_size
        self.batch_size = batch_size
        self.input_n = input_n

    def SortPrep(self):
        X = self.X
        Y = self.Y
        X_train, X_test, Y_train, Y_test = model_selection.train_test_split(
            X, Y, test_size=test_size, random_state=123
        )
        return X_train, Y_train, X_test, Y_test

In [ ]:
test_size = 10**4
batch_size = len(m)
input_n = 35
NN1 = LearningHiggs(data, m, test_size, batch_size, input_n)
X_train, Y_train, X_test, Y_test = NN1.SortPrep()

fig = plt.figure(figsize=(20, 30))
fig.tight_layout()
for b in range(len(branches)):
    ax = fig.add_subplot(9, 4, 1 + b if b < 8 else 2 + b)
    plt.subplots_adjust(hspace=0.3, wspace=0.5)
    ax.hist(X_train[branches[b]])
    ax.set_xlabel(branches[b])
    ax.set_ylabel("Events/Bin")

### Data Expectations

As I mentioned before, the machine will figure out the relationship between the features solely from referencing the target data, making this a regression problem. The data that I will be inputting is called *data* and it can be seen in the data frame table in the previous subsection, while the target data is the parameter *m*. I could also calculate the mass from the input data and have that be the target data, programming Equation (6), but I much rather use the files that were already given to me.

When the machine is done training the output will be mass, and then I will need to plot the mass distribution for the events. Ideally, the mass should be fairly close to the actual value, 125.11 GeV, but it's preferred that it does not overestimate. If the machine were to just output the actual output, it would imply that it isn't learning anything and it just "memorized" the actual value, so in order to avoid that I won't input the actual value into my algorithm.

## Neural Network

To tackle this problem I decided to brute force this problem and use a fully connected Neural Network (NN). The NN consists of nine layers, and the final output will go through a Sigmoid Function to ensure that my output is strictly positive. The activation function that I used in every layer was a hyperbolic tangent function, and the reason for this is very simple, this problem is *non*-linear, so I used that as a way of wrangling the data. I should also mention, that because this is a regression problem I used the MSE Loss Function since it is the standard function used for this kind of problems. Finally, after various attempts I decided to use the SGD optimizer since it allowed for the use of momentum so that the loss cuves would converge must faster.

### Network

In order to keep things in a single code block I wrote a class that consists of the following functions:
+ __init__: Initializes the variables that will be used within the class.
+ SortPrep: Sorts the data so that it will in its respective variables; training or testing.
+ Tensor_Network_Setup: Creates the test and training tensors, houses the net used, and is responsible for the the training.
    + The values for the nodes was nothing special I just wanted to increase the number of nodes and then slowly start decreasing them.
+ plottingStuff: This takes care of plotting either the loss curve or the loss curve and the mass distribution.

In [ ]:
class LearningHiggs:
    def __init__(self, X, Y, test_size, batch_size, input_n):
        self.X = X
        self.Y = Y
        self.test_size = test_size
        self.batch_size = batch_size
        self.input_n = input_n

    def SortPrep(self):
        X = self.X
        Y = self.Y
        X_train, X_test, Y_train, Y_test = model_selection.train_test_split(
            X, Y, test_size=test_size, random_state=123
        )
        return X_train, Y_train, X_test, Y_test

    # Function that does most of the work <- Reword this
    def Tensor_Network_Setup(
        self, X_train, Y_train, X_test, Y_test, m, pathfile, lr, momentum, n_epochs
    ):
        torch.manual_seed(123)
        sc = StandardScaler()
        sc2 = StandardScaler()
        X_train, X_test = sc.fit_transform(X_train), sc.transform(X_test)
        Y_train, Y_test = sc2.fit_transform(Y_train), sc2.transform(Y_test)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float)
        XY_train = torch.utils.data.TensorDataset(X_train_tensor, Y_train_tensor)
        loader = torch.utils.data.DataLoader(XY_train, batch_size=len(m), shuffle=True)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float)
        Y_test_tensor = torch.tensor(Y_test, dtype=torch.float)
        # Setting my net.
        net = torch.nn.Sequential(
            torch.nn.Linear(self.input_n, 100),
            torch.nn.Tanh(),
            torch.nn.Linear(100, 75),
            torch.nn.Tanh(),
            torch.nn.Linear(75, 60),
            torch.nn.Tanh(),
            torch.nn.Linear(60, 50),
            torch.nn.Tanh(),
            torch.nn.Linear(50, 40),
            torch.nn.Tanh(),
            torch.nn.Linear(40, 30),
            torch.nn.Tanh(),
            torch.nn.Linear(30, 20),
            torch.nn.Tanh(),
            torch.nn.Linear(20, 10),
            torch.nn.Tanh(),
            torch.nn.Linear(10, 1),
            torch.nn.Sigmoid(),
        )
        torch.save(net.state_dict(), pathfile)
        optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=momentum)
        loss_fn = torch.nn.MSELoss()
        # Empty array for the losses of the training and test data respectively
        losses = []
        losses_test = []
        # Reloading the previously saved state file, net.pth
        net.load_state_dict(torch.load(pathfile))
        for epoch in range(n_epochs):
            # Began training
            net.train()
            for x_batch, y_batch in loader:
                Y_pred = net(x_batch)
                optimizer.zero_grad()
                loss = loss_fn(Y_pred, y_batch)
                loss.backward()
                optimizer.step()
            # Appended the loss of the training data
            losses.append(loss.data)
            net.eval()
            # Setting the prediction values from the x_test_tensor
            Y_pred = net(X_test_tensor)
            # Appending the losses of the test data
            losses_test = np.append(losses_test, loss_fn(Y_pred, Y_test_tensor).data)
        Y_predN = sc2.inverse_transform(Y_pred.detach())
        return losses, losses_test, Y_predN

    def plottingStuff(self, losses, losses_test, Y_predN, hist_loss):
        if hist_loss == False:
            # Stuff to plot
            plt.plot(losses, ".", label="Train")
            plt.plot(losses_test, ".", label="Test")
            plt.legend()
            plt.title("Loss Curve", size=15)
            plt.xlabel("Training Epoch", size=15)
            plt.ylabel("MSE Loss", size=15)
            plt.yscale("log")
            plt.show()
        else:
            # plots both the mass distribution and the loss curve
            plt.plot(losses, ".", label="Train")
            plt.plot(losses_test, ".", label="Test")
            plt.legend()
            plt.title("Loss Curve", size=15)
            plt.xlabel("Training Epoch", size=15)
            plt.ylabel("MSE Loss", size=15)
            plt.yscale("log")
            sigma = np.std(Y_predN)
            average = np.mean(Y_predN)
            print("Average and Standard Deviation:", average, "\u00b1", sigma)
            print("Relative Error:", (np.abs(average - 125.11) / 125.11) * 100, "%")
            plt.hist(Y_predN)
            plt.xlabel("Invariant Mass [GeV]", size=15)
            plt.ylabel("Number of Events", size=15)
            plt.title("$H\\to\mu^{+}\mu^{-}$", size=15)
            plt.show()

### Hyper-parameters

#### Layers and Nodes

The layers and nodes were chosen randomly. The number of layers were chosen to filter the data. I decided to treat it like a data grater; so that the first layer had a large number of nodes and they decreased in multiples of 10 or 5. I did it this way so that it resembled the usual schematics that were displayed in lecture.

#### Learning Rate

For this project, since I knew that the problem was non-linear and required careful computation I figured I should set the learning rate really low, so for the final calculations I set the learning rate to be 0.01. However, for the cases in which I had the machine produce loss curves to see how the momentum would affect the loss curve I set the learning rate to 0.1.

#### Momentum

Using the NN shown in the previous section, I played with the hyper-parameters. First I focused on using different values of momentum, for a total number of four trials within the interval [0.1, 0.9]. The number of epochs used for the trials was kept at a value of 20, while the learning rate was kept at 0.1.

In [ ]:
test_size = 10**4
batch_size = len(m)
input_n = 35
NN1 = LearningHiggs(data, m, test_size, batch_size, input_n)
X_train, Y_train, X_test, Y_test = NN1.SortPrep()

pathfile = "netP1.pth"
lr = 0.1
n_epochs = 100

momentum = np.linspace(0.1, 0.9, 4)
LT1 = []
LTT1 = []
for p in momentum:
    losses, losses_test, Y_predN = NN1.Tensor_Network_Setup(
        X_train, Y_train, X_test, Y_test, m, pathfile, lr, p, n_epochs
    )
    LT1.append(losses)
    LTT1.append(losses_test)

fig, z = plt.subplots(2, 2, figsize=(9, 5))
z[0, 0].plot(LT1[0], ".", label="Train")
z[0, 0].plot(LTT1[0], ".", label="Test")
z[0, 0].set_title("Loss Curve: Momentum={}".format("%.2f" % momentum[0]))
z[0, 0].legend()
z[0, 1].plot(LT1[1], ".", label="Train")
z[0, 1].plot(LTT1[1], ".", label="Test")
z[0, 1].set_title("Loss Curve: Momentum={}".format("%.2f" % momentum[1]))
z[0, 1].legend()
z[1, 0].plot(LT1[2], ".", label="Train")
z[1, 0].plot(LTT1[2], ".", label="Test")
z[1, 0].set_title("Loss Curve: Momentum={}".format("%.2f" % momentum[2]))
z[1, 0].legend()
z[1, 1].plot(LT1[3], ".", label="Train")
z[1, 1].plot(LTT1[3], ".", label="Test")
z[1, 1].set_title("Loss Curve: Momentum={}".format("%.2f" % momentum[3]))
z[1, 1].legend()
fig.tight_layout()
plt.xlabel("Training Epoch")
plt.ylabel("MSE Loss")
plt.yscale("log")
plt.show()

Based on the set of curves I decided to keep the momentum value set to 0.2 since it seems to be very close to the train data until around 20 epochs for the first two plots.

#### Final Hyper-Parameters

Based on all of this information I decided to keep the following hyper-parameters:
+ Learning Rate: 0.01
+ Momentum: 0.2
+ Epochs: 1,000
+ Test Size: 10,000
+ Batch Size: 49,999
+ Input number: 35

## Learning with Hyper-Parameters

As mentioned in the previous section, once the final set of hyper-parameters were chosen I began to train my machine and then using its predicted values to plot the mass distribution. Using the mass distribution histogram, I would be able to determine how well my machine learned to calculate the Higgs boson's invariant mass.

In [ ]:
# Name of the input data file:
dataFile = "lowlevelAna_Hmumu.hf5"
# Branches that are of interest to me:
branches = (
    "numjet",
    "numlepton",
    "numbtagjet",
    "met",
    "metphi",
    "lepton1pT",
    "lepton1eta",
    "lepton1phi",
    "lepton2pT",
    "lepton2eta",
    "lepton2phi",
    "jet1pT",
    "jet1eta",
    "jet1phi",
    "jet1b",
    "jet2pT",
    "jet2eta",
    "jet2phi",
    "jet2b",
    "jet3pT",
    "jet3eta",
    "jet3phi",
    "jet3b",
    "jet4pT",
    "jet4eta",
    "jet4phi",
    "jet4b",
    "jet5pT",
    "jet5eta",
    "jet5phi",
    "jet5b",
    "jet6pT",
    "jet6eta",
    "jet6phi",
    "jet6b",
)
# Target data file; true masses.
targetFile = "targets_Hmumu.txt"
# Initializing my ReadsData object:
trialTwo = ReadsData(dataFile, branches, targetFile)
# Defining the variables that will hold the output of DataRead()
data, m = trialTwo.DataRead()


test_size = 10**4
batch_size = len(m)
input_n = 35
NN3 = LearningHiggs(data, m, test_size, batch_size, input_n)
X_train, Y_train, X_test, Y_test = NN3.SortPrep()

pathfile = "netP3.pth"
lr = 0.01
n_epochs = 1000
momentum = 0.2
lossesN, losses_testN, Y_predN = NN3.Tensor_Network_Setup(
    X_train, Y_train, X_test, Y_test, m, pathfile, lr, momentum, n_epochs
)
plt.plot(lossesN, ".", label="Train")
plt.plot(losses_testN, ".", label="Test")
plt.legend()
plt.title("Loss Curve", size=15)
plt.xlabel("Training Epoch", size=15)
plt.ylabel("MSE Loss", size=15)
plt.yscale("log")
plt.show()

sigma = np.std(Y_predN)
average = np.mean(Y_predN)
print("Average and Standard Deviation:", average, "\u00b1", sigma)
print("Percent Error:", (np.abs(average - 125.11) / 125.11) * 100, "%")
plt.hist(Y_predN)
plt.xlabel("Invariant Mass [GeV]", size=15)
plt.ylabel("Number of Events", size=15)
plt.title("$H\\to\mu^{+}\mu^{-}$", size=15)
plt.show()

As we can see the machine has indeed learned how to calculate the invariant mass of the Higgs boson. Therefore it has learned everything we covered in the Particle Physics Primer. Although the mass is lower than the actual value, its preferred that it produces this calculations because it shows that it has actually learned and not just memorized what the actual mass is.

## Conclusions

Based on the figures, the machine has indeed learned a non-linear problem, and even though the value that the machine calculated is lower than the actual mass of the Higgs boson, it shows that there is room for improvement within the algorithm that was used. The power of ML within Particle Physics is indeed noted, especially in cases like this where the machine was able to learn not just relativistic kinematics, but also about collider coordinates; this is no small feat as these are advanced topics within Physics.

A potential change that can be made to the algorithm includes having the machine look back at the values that it produces and find the discrepencies between its calculated values and the actual mass of the Higgs boson and make the appropriate corrections. Also, I could make this algorithm better since at the very end it would not produce the output that I was finally able to show. In order to get it to produce the final output, I had to write out the code for the plots instead of using my predefined class function.

Finally, the applications of ML within Particle Physics can be extended further such that there is unsupervised learning. In those cases the machine is actively cycling through data looking for correlations, and being able to flag anomolies. We call this, Unsupervised Anomoly Detection, and it is an active topic within Particle Physics as it frees up man power [[6]](https://arxiv.org/abs/2110.06948).

## References

[[1]](https://hdsr.mitpress.mit.edu/pub/xqle7lat/release5#:~:text=Machine%20learning%20(ML%20has%20played,with%20a%20primordial%20bottom%20quark.) Machine Learning within Particle Physics

[[2]](https://iopscience.iop.org/article/10.1088/1742-6596/1085/2/022008/pdf) Machine Learning and Mass Reconstruction

[[3]](https://mikefragugliacom.files.wordpress.com/2016/12/introduction-to-elementary-particles-gnv64.pdf) Introduction to Particle Physics

[[4]](https://iopscience.iop.org/book/mono/978-0-7503-2444-1) Practical Collider Physics

[[5]](https://en.wikipedia.org/wiki/Invariant_mass#Collider_experiments) Collider Experiments

[[6]](https://arxiv.org/abs/2110.06948) Unsupervised Anomoly Detection